# 🔍 RAG LLM System — Pipeline Demo

**Stack:** LangChain + Pinecone + BAAI/bge-small-en-v1.5 + flan-t5-base  
**Documents:** Wikipedia articles + ArXiv research papers + PDF files  
**Key feature:** Confidence-based routing — cosine score + cross-encoder reranker

## What is RAG?

Large Language Models have a knowledge cutoff and can hallucinate. RAG (Retrieval-Augmented Generation) grounds the LLM's answers in real documents:

$$\text{Answer} = \text{LLM}(\text{question} + \text{retrieved context})$$

Instead of relying on parametric memory (weights), the model retrieves relevant passages from a document store and uses them as context.

## The Problem with Naive RAG

Most RAG systems blindly pass retrieved chunks to the LLM regardless of relevance. When the retrieved context is irrelevant, the LLM tries to answer anyway — producing **confident wrong answers** (hallucination).

## Our Solution — Confidence-Based Routing

```
Question → Retrieve top-k chunks → Cosine score check
  score ≥ 0.75  → RAG answer (clearly relevant)
  score < 0.50  → LLM answer (clearly irrelevant)
  score 0.50-0.75 → Cross-encoder reranker → final decision
```

## Architecture

| Component | Model | Purpose |
|---|---|---|
| Embedder | BAAI/bge-small-en-v1.5 | Embed chunks + queries (384-dim) |
| Vector DB | Pinecone | Persistent vector storage + search |
| Reranker | cross-encoder/ms-marco-MiniLM-L-6-v2 | Borderline score refinement |
| Generator | google/flan-t5-base | Answer generation |
| Router | Custom logic | Confidence-based RAG vs LLM routing |

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(config_path='../configs/rag_config.yaml')
print('Pipeline ready.')

## Step 1 — Ingest Documents

We ingest three document types to build a mixed knowledge base:
- Wikipedia article on Transformers
- ArXiv paper — Attention Is All You Need
- Wikipedia article on Indian Constitution (legal domain)

In [ ]:
# Ingest Wikipedia — Transformer architecture
result = pipeline.ingest(
    source   = 'Transformer (machine learning model)',
    doc_type = 'wikipedia',
)
print(result)

In [ ]:
# Ingest ArXiv — Attention Is All You Need (1706.03762)
result = pipeline.ingest(
    source   = '1706.03762',
    doc_type = 'arxiv',
)
print(result)

In [ ]:
# Ingest Wikipedia — Legal domain (Indian Constitution)
result = pipeline.ingest(
    source   = 'Constitution of India',
    doc_type = 'wikipedia',
)
print(result)

## Step 2 — Query with RAG Route

These questions are directly answerable from the indexed documents.
Expected route: `rag` with high cosine score.

In [ ]:
result = pipeline.query('What is the self-attention mechanism in transformers?')

print(f"Answer       : {result['answer']}")
print(f"Route        : {result['route_used']}")
print(f"Cosine score : {result['cosine_score']}")
print(f"Rerank score : {result['rerank_score']}")
print(f"Reason       : {result['routing_reason']}")
print(f"Sources      : {result['source_documents']}")
print(f"Latency      : {result['latency_ms']} ms")

In [ ]:
result = pipeline.query('How many articles are in the Indian Constitution?')

print(f"Answer       : {result['answer']}")
print(f"Route        : {result['route_used']}")
print(f"Cosine score : {result['cosine_score']}")
print(f"Sources      : {result['source_documents']}")

## Step 3 — Query with LLM Route

These questions are NOT in the indexed documents.
Expected route: `llm` with low cosine score — system falls back to model knowledge.

In [ ]:
result = pipeline.query('What is the capital of Australia?')

print(f"Answer       : {result['answer']}")
print(f"Route        : {result['route_used']}")
print(f"Cosine score : {result['cosine_score']}")
print(f"Reason       : {result['routing_reason']}")

## Step 4 — Borderline Query (Reranker Triggered)

A question that partially matches the indexed content.
Expected: cosine score in 0.50-0.75 range → reranker triggered.

In [ ]:
result = pipeline.query('What are the key components of a neural network encoder?')

print(f"Answer       : {result['answer']}")
print(f"Route        : {result['route_used']}")
print(f"Cosine score : {result['cosine_score']}")
print(f"Rerank score : {result['rerank_score']}")
print(f"Reason       : {result['routing_reason']}")

## Step 5 — Routing Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt

stats = pipeline.logger.get_stats()
print('Query Stats:', stats)

if stats['total_queries'] > 0:
    labels  = ['RAG / Reranked', 'LLM Only']
    values  = [stats['rag_route'], stats['llm_route']]
    colors  = ['#1D9E75', '#E85D24']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax1.set_title('Routing Distribution')

    logs    = pipeline.logger.get_recent(n=100)
    scores  = [l['cosine_score'] for l in logs]
    routes  = [l['route_used'] for l in logs]
    colors2 = ['#1D9E75' if r in ('rag','reranked') else '#E85D24' for r in routes]

    ax2.scatter(range(len(scores)), scores, c=colors2, alpha=0.7, s=40)
    ax2.axhline(y=0.75, color='green', linestyle='--', alpha=0.7, label='RAG threshold')
    ax2.axhline(y=0.50, color='red',   linestyle='--', alpha=0.7, label='LLM threshold')
    ax2.set_xlabel('Query index')
    ax2.set_ylabel('Cosine score')
    ax2.set_title('Cosine Scores per Query')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.suptitle('RAG System — Routing Analysis', fontsize=13)
    plt.tight_layout()
    os.makedirs('../assets', exist_ok=True)
    plt.savefig('../assets/routing_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → assets/routing_analysis.png')

## Step 6 — View Recent Query Logs

In [ ]:
import json

logs = pipeline.logger.get_recent(n=5)
for log in logs:
    print(f"Q: {log['question']}")
    print(f"A: {log['answer'][:100]}...")
    print(f"Route: {log['route_used']} | Cosine: {log['cosine_score']} | Latency: {log['latency_ms']}ms")
    print('-' * 60)

## Results Summary

| Query Type | Route | Cosine Score | Rerank Score |
|---|---|---|---|
| Transformer self-attention | rag | *fill* | — |
| Indian Constitution articles | rag | *fill* | — |
| Capital of Australia | llm | *fill* | — |
| Neural network encoder | reranked | *fill* | *fill* |

## Key Takeaways

**Confidence routing prevents hallucination** — when cosine score is low the system correctly falls back to LLM knowledge rather than generating a wrong answer from irrelevant context.

**Two-stage scoring is more reliable** — cosine similarity is fast but imprecise in the borderline zone. The cross-encoder reranker sees query and passage together, catching relevance that bi-encoder misses.

**Mixed domain knowledge base works** — the same system answers questions about ML research, legal documents, and general knowledge by routing each query appropriately.